In [0]:
%pip install --upgrade \
    "unitycatalog-langchain[databricks]" \
    databricks-langchain \
    langgraph \
    mlflow

In [0]:
%pip install --force-reinstall \
    "langgraph==1.0.10" \
    "langgraph-prebuilt==1.0.8"

In [0]:
dbutils.library.restartPython()

In [0]:
from importlib.metadata import version

for package in [
    "unitycatalog-langchain",
    "unitycatalog-ai",
    "databricks-langchain",
    "langgraph",
    "langgraph-prebuilt",
    "langgraph-checkpoint",
    "langchain-core",
    "mlflow"
]:
    try:
        print(package, version(package))
    except Exception:
        print(package, "Not installed")

In [0]:
from databricks_langchain import ChatDatabricks, UCFunctionToolkit
from unitycatalog.ai.core.databricks import DatabricksFunctionClient
from langgraph.prebuilt import create_react_agent
import mlflow

print("All imports successful!")

### R1 - Tools as UC functions

In [0]:
%sql
-- Create the Unity Catalog function
CREATE OR REPLACE FUNCTION claims.genai.lookup_claim(
    claim_id STRING
)
RETURNS TABLE (
    claim_id STRING,
    member_id STRING,
    hospital_id STRING,
    status STRING,
    claim_type STRING,
    admission_date DATE,
    discharge_date DATE,
    billed_amount DOUBLE,
    approved_amount DOUBLE,
    rejection_code STRING,
    is_pre_existing BOOLEAN,
    plan_code_at_admission STRING,
    plan_family STRING,
    room_rent_per_day INT,
    room_rent_cap_per_day INT,
    room_rent_over_cap BOOLEAN,
    policy_start_date DATE,
    policy_tenure_months INT,
    days_since_policy_start INT,
    is_network_at_admission BOOLEAN
)
COMMENT 'Returns stored claim facts from claims.gold.claims_enriched for a given claim ID, plus days between policy start and admission. Does not make payout decisions.'
RETURN
    SELECT
        c.claim_id,
        c.member_id,
        c.hospital_id,
        c.status,
        c.claim_type,
        c.admission_date,
        c.discharge_date,
        c.billed_amount,
        c.approved_amount,
        c.rejection_code,
        c.is_pre_existing,
        c.plan_code_at_admission,
        c.plan_family,
        c.room_rent_per_day,
        c.room_rent_cap_per_day,
        c.room_rent_over_cap,
        c.policy_start_date,
        c.policy_tenure_months,

        DATEDIFF(
            c.admission_date,
            c.policy_start_date
        ) AS days_since_policy_start,

        c.is_network_at_admission

    FROM claims.gold.claims_enriched AS c
    WHERE c.claim_id = lookup_claim.claim_id;

In [0]:
%sql
-- Create the function claims.genai.plan_at 
CREATE OR REPLACE FUNCTION claims.genai.plan_at(
    member_id STRING,
    as_of_date DATE
)
RETURNS TABLE (
    member_id STRING,
    policy_id STRING,
    plan_code STRING,
    plan_family STRING,
    room_rent_cap_per_day INT,
    copay_pct INT,
    ped_waiting_months INT,
    initial_waiting_days INT,
    valid_from DATE,
    valid_to DATE
)
COMMENT 'Returns the member policy version and corresponding plan terms valid on a specified date. Uses policy history and plan terms; does not make claim payout decisions.'
RETURN
    SELECT
        p.member_id,
        p.policy_id,
        p.plan_code,
        pl.plan_family,
        CAST(pl.room_rent_cap_per_day AS INT) AS room_rent_cap_per_day,
        CAST(pl.copay_pct AS INT) AS copay_pct,
        CAST(pl.ped_waiting_months AS INT) AS ped_waiting_months,
        CAST(pl.initial_waiting_days AS INT) AS initial_waiting_days,
        CAST(p.__START_AT AS DATE) AS valid_from,
        CAST(p.__END_AT AS DATE) AS valid_to
    FROM claims.silver.policy_scd2 AS p
    INNER JOIN claims.raw.plans AS pl
        ON p.plan_code = pl.plan_code
    WHERE p.member_id = plan_at.member_id
      AND CAST(p.__START_AT AS DATE) <= plan_at.as_of_date
      AND (
            p.__END_AT IS NULL
            OR CAST(p.__END_AT AS DATE) > plan_at.as_of_date
      );

In [0]:
%sql
-- Create the function claims.genai.search_policy
CREATE OR REPLACE FUNCTION claims.genai.search_policy(
    query STRING
)
RETURNS TABLE (
    chunk_id STRING,
    doc_name STRING,
    heading STRING,
    text STRING
)
COMMENT 'Searches the policy binder using hybrid keyword and semantic search. Returns up to five most relevant policy chunks, with chunk ID, document name, heading, and text. Use this tool to retrieve policy evidence relevant to a question.'
RETURN
    SELECT
        CAST(v.chunk_id AS STRING) AS chunk_id,
        CAST(v.doc_name AS STRING) AS doc_name,
        CAST(v.heading AS STRING) AS heading,
        CAST(v.text AS STRING) AS text
    FROM vector_search(
        index => 'claims.genai.policy_chunks_idx',
        query_text => query,
        query_type => 'HYBRID',
        num_results => 5
    ) AS v;

In [0]:
%sql
-- Create the function claims.genai.compute_payable
CREATE OR REPLACE FUNCTION claims.genai.compute_payable(
  plan_code STRING,
  is_pre_existing BOOLEAN,
  policy_tenure_months INT COMMENT 'Completed months from the member''s FIRST policy start to admission (PED waiting basis)',
  days_since_policy_start INT COMMENT 'Days from the member''s FIRST policy start to admission (initial waiting basis)',
  is_network_at_admission BOOLEAN,
  room_rent_per_day INT,
  billed_amount DOUBLE
)
RETURNS STRING
COMMENT 'Decides whether a claim is payable and for how much, using only the rules in the Definitions (waiting periods, room-rent cap deduction, co-pay). Pure deterministic lookup and arithmetic — never calls a model. Returns a JSON string: plan_code, room_rent_cap_per_day, room_rent_over_cap, waiting_period_ok, payable, network_at_admission, room_rent_deduction_pct, copay_pct, reason_code, payable_amount.'
RETURN
  WITH plan AS (
    -- MAX() makes the correlated plan lookup return at most one row (raw.plans has exactly one row per plan_code)
    SELECT MAX(room_rent_cap_per_day) AS room_rent_cap_per_day,
           MAX(copay_pct) AS base_copay_pct,
           MAX(ped_waiting_months) AS ped_waiting_months,
           MAX(initial_waiting_days) AS initial_waiting_days
    FROM claims.raw.plans
    WHERE plan_code = compute_payable.plan_code
  ),
  checks AS (
    SELECT
      p.room_rent_cap_per_day,
      (compute_payable.room_rent_per_day > p.room_rent_cap_per_day) AS room_rent_over_cap,
      (compute_payable.days_since_policy_start >= p.initial_waiting_days) AS initial_waiting_ok,
      (NOT compute_payable.is_pre_existing OR compute_payable.policy_tenure_months >= p.ped_waiting_months) AS ped_waiting_ok,
      CASE
        WHEN compute_payable.room_rent_per_day <= p.room_rent_cap_per_day THEN 0.0
        ELSE ROUND((1 - (p.room_rent_cap_per_day * 1.0 / compute_payable.room_rent_per_day)) * 100, 2)
      END AS room_rent_deduction_pct,
      p.base_copay_pct + CASE WHEN compute_payable.is_network_at_admission THEN 0 ELSE 20 END AS copay_pct
    FROM plan p
  ),
  decided AS (
    SELECT *,
      (initial_waiting_ok AND ped_waiting_ok) AS waiting_period_ok,
      (initial_waiting_ok AND ped_waiting_ok) AS payable
    FROM checks
  ),
  amounts AS (
    SELECT *,
      CASE WHEN payable THEN
        ROUND(
          (compute_payable.billed_amount * (1 - room_rent_deduction_pct / 100.0))
          * (1 - copay_pct / 100.0),
        2)
      ELSE 0.0 END AS payable_amount,
      CASE
        WHEN NOT initial_waiting_ok THEN 'INITIAL_WAITING'
        WHEN NOT ped_waiting_ok THEN 'PED_WAITING'
        WHEN (room_rent_deduction_pct > 0 OR copay_pct > 0) THEN 'PAYABLE_WITH_DEDUCTION'
        ELSE 'PAYABLE_IN_FULL'
      END AS reason_code
    FROM decided
  )
  SELECT to_json(named_struct(
    'plan_code', compute_payable.plan_code,
    'room_rent_cap_per_day', room_rent_cap_per_day,
    'room_rent_over_cap', room_rent_over_cap,
    'waiting_period_ok', waiting_period_ok,
    'payable', payable,
    'network_at_admission', compute_payable.is_network_at_admission,
    'room_rent_deduction_pct', room_rent_deduction_pct,
    'copay_pct', copay_pct,
    'reason_code', reason_code,
    'payable_amount', payable_amount
  ))
  FROM amounts;

In [0]:
%sql
SHOW USER FUNCTIONS IN claims.genai;

In [0]:

import mlflow
from pathlib import Path

from databricks_langchain import ChatDatabricks, UCFunctionToolkit
from unitycatalog.ai.core.databricks import DatabricksFunctionClient
from langgraph.prebuilt import create_react_agent


# ---------------- Configuration ----------------

MODEL_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

BASE_PATH = Path(
    "/Workspace/Users/surajitm0nd0l12@gmail.com/"
    "Saffron_Shield/Saffron_Shield_dabs/agents"
)

SYSTEM_PROMPT = (BASE_PATH / "prompts" / "system_v1.md").read_text()

EXPERIMENT = (
    "/Users/surajitm0nd0l12@gmail.com/Saffron_Shield_R2"
)

# Enable MLflow tracing before running the agent
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment(EXPERIMENT)
mlflow.langchain.autolog()

# Unity Catalog function client
uc_client = DatabricksFunctionClient()

# LLM
llm = ChatDatabricks(
    endpoint=MODEL_ENDPOINT,
    temperature=0
)


# ---------------- Tool Routing ----------------

FUNCTIONS = {
    "lookup": "claims.genai.lookup_claim",
    "plan": "claims.genai.plan_at",
    "policy": "claims.genai.search_policy",
    "payout": "claims.genai.compute_payable"
}


def get_allowed_tools(question):

    q = question.lower()

    # Claim status: lookup claim + historical plan
    if "status" in q and "claim" in q:
        return [FUNCTIONS["lookup"], FUNCTIONS["plan"]]

    # Policy questions: policy search only
    if any(word in q for word in [
        "policy", "waiting period", "pre-existing",
        "ped", "coverage", "exclusion"
    ]):
        return [FUNCTIONS["policy"]]

    # Other questions: all tools available
    return list(FUNCTIONS.values())


# ---------------- Agent Execution ----------------

def run_agent(question):

    allowed_functions = get_allowed_tools(question)

    toolkit = UCFunctionToolkit(
        function_names=allowed_functions,
        client=uc_client
    )

    agent = create_react_agent(
        model=llm,
        tools=toolkit.tools,
        prompt=SYSTEM_PROMPT
    )

    result = agent.invoke({
        "messages": [
            {"role": "user", "content": question}
        ]
    })

    # Extract actual tool calls from LangGraph messages
    tool_calls = []

    for message in result["messages"]:
        for call in getattr(message, "tool_calls", []):
            tool_calls.append({
                "name": call["name"],
                "args": call["args"]
            })

    final_answer = result["messages"][-1].content

    # Flush and retrieve the MLflow trace ID
    mlflow.flush_trace_async_logging()
    trace_id = mlflow.get_last_active_trace_id()

    print("Question:", question)
    print("Tool calls:", tool_calls)
    print("Final answer:", final_answer)
    print("MLflow trace ID:", trace_id)

    return {
        "trace_id": trace_id,
        "tool_calls": tool_calls,
        "final_answer": final_answer,
        "messages": result["messages"]
    }


In [0]:
status_result = run_agent(
    "What's the status of claim CLM-10042?"
)

In [0]:
policy_result = run_agent(
    "What is the pre-existing disease (PED) waiting period for a corporate group plan?"
)